# Intyoza AI service (Kaggle)

Runs Intyoza's trained answer model on Kaggle's GPU and gives the backend a fixed address. Nothing is trained here.
New questions and answers you added to `Intyoza_Dataset.xlsx` (dataset `intyoza-data`) are answered right away.

**Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.

**Inputs** (Add Input > Your Work):
- Datasets: `intyoza` (code + matcher), `intyoza-code` (latest code), `intyoza-data` (your data)
- Notebooks: the training notebook (its output `intyoza_outputs` holds the trained model)

**Secrets** (Add-ons > Secrets, once; tick them for this notebook):
- `INTYOZA_API_KEY`: your fixed service key
- `NGROK_AUTHTOKEN` and `NGROK_DOMAIN`: from ngrok.com, for a fixed address (without them the address changes every run)

Voice input (Kinyarwanda speech to text) is set up in step 2b; set `VOICE = False` there to skip it.

Then **Run All** and keep this tab open while you test.

## 1. Setup

In [ ]:
import os, sys, glob
ks = []
try:   # latest code straight from GitHub (Kaggle secrets INTYOZA_REPO, INTYOZA_BRANCH, GITHUB_TOKEN): no zip uploads
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    def _get(name):
        try:
            return (_s.get_secret(name) or "").strip()
        except Exception:
            return ""
    _repo, _branch, _tok = _get("INTYOZA_REPO"), _get("INTYOZA_BRANCH") or "main", _get("GITHUB_TOKEN")
    if _repo:
        import subprocess, shutil
        shutil.rmtree("/tmp/intyoza_repo", ignore_errors=True)
        _url = f"https://{'x-access-token:' + _tok + '@' if _tok else ''}github.com/{_repo}.git"
        _r = subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", _branch, _url, "/tmp/intyoza_repo"],
                            capture_output=True, text=True)
        ks = glob.glob("/tmp/intyoza_repo/ai/intyoza_answer_model/kaggle_setup.py")
        if not ks:
            print("Code from GitHub failed:", (_r.stderr.replace(_tok, "***") if _tok else _r.stderr)[-200:])
except ImportError:
    pass
ks = ks or glob.glob("/kaggle/input/**/kaggle_setup.py", recursive=True)
assert ks, "Set the Kaggle secret INTYOZA_REPO (or add the dataset intyoza-code)."
sys.path.insert(0, os.path.dirname(ks[0]))
import kaggle_setup
FOLDER = kaggle_setup.setup()
assert os.path.exists("models/intyoza_writer/adapter_config.json"), "Trained model not found: add the training notebook's output as input."
assert os.path.exists("models/intyoza_matcher"), "Matcher not found: add the dataset 'intyoza' (it has models/)."
!pip -q install -U sentence-transformers accelerate pypdf peft bitsandbytes fastapi uvicorn requests
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: turn on GPU T4 x2 in Settings")

## 2. Load the model (a few minutes)

In [ ]:
os.environ.setdefault("INTYOZA_DEMO", "1")      # "0" when Intyoza goes live: approved data only
import intyoza_core, doc_engine, chatbot, answer_engine, get_documents, demo_data
LOG = os.path.join(FOLDER, "intyoza_unanswered.csv")
bot = intyoza_core.load_bot("models", ".", use_documents=False, online=True, log_path=LOG)
get_documents.prepare(bot.enc, "documents", "models/doc_index")      # adds new documents to the index
docs = doc_engine.DocEngine(bot.enc, "models/doc_index", use_llm=False, use_translator=False)
llm = answer_engine.make_llm(answer_engine.WRITER_DIR)
assert llm is not None, "The answer model did not load (is the GPU on?)."
extra = demo_data.load() if demo_data.enabled() else None
engine = answer_engine.AnswerEngine(bot, llm, docs=docs, doc_dir="documents", log_path=LOG, demo=extra)
chat = chatbot.IntyozaChat(bot, docs=docs, engine=engine)
print(f"Loaded: {len(bot.qa)} approved questions, {len(docs.passages)} document passages, model = {llm.name}")

## 2b. Voice (optional)

Kinyarwanda speech to text on this GPU (NVIDIA NeMo `stt_rw_conformer_transducer_large`, CC-BY-4.0): the app can send a
recorded question to `POST /api/transcribe` and gets the text back. No voice leaves this service.
Installing NeMo takes a few minutes. Set `VOICE = False` to skip it; the chat works either way.

In [ ]:
VOICE = True
stt = None
if VOICE:
    import subprocess, sys
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nemo_toolkit[asr]"], capture_output=True, text=True)
    try:
        import speech
        stt = speech.SpeechToText()
        stt.load("rw")                       # downloads the Kinyarwanda speech model now, not on the first request
        print("Voice ready:", stt.status())
    except Exception as e:
        print("Voice NOT available (the chat still works):", str(e)[:300])
        print(r.stderr[-800:])
        stt = None

## 2c. Training conversations from the Phrasings tab (optional, off by default)

`FROM_PHRASINGS = True` writes training conversations with this model before the service starts: the young person's
messages are real ways of asking from the Phrasings tab (2 in English and 1 in Kinyarwanda per approved question,
training split only, no crisis questions), and only replies that pass every check are kept. Run it with
**Save Version > Save & Run All**; it stops after `TIME_LIMIT_H` hours and keeps what it wrote. The result,
`Intyoza_Conversations_From_Phrasings.xlsx`, is in the **Output** tab, with the Conversations tab's columns (Review
status: Draft) and a random 10% marked in **Spot check**. Leave `FROM_PHRASINGS = False` for the normal service.

In [ ]:
FROM_PHRASINGS = False
if FROM_PHRASINGS:
    import writer_training
    TIME_LIMIT_H = 3          # stops after this many hours and keeps what it wrote
    OUT = "/kaggle/working/Intyoza_Conversations_From_Phrasings.xlsx"
    rows = writer_training.conversations_from_phrasings(chat, folder=".", out_path=OUT, n=1100, time_limit_h=TIME_LIMIT_H)
    raise SystemExit("Done: download Intyoza_Conversations_From_Phrasings.xlsx from the Output tab.")

## 3. Start the service

In [ ]:
import public_service
url, key = public_service.start(chat, stt=stt)

## 4. Test it like the backend does

In [ ]:
import requests
for q in ["Agakingirizo ni iki?", "Can I get pregnant the first time?"]:
    r = requests.post(f"{url}/api/chat", json={"message": q}, timeout=180).json()
    print("Q:", q, "\nA:", r["answer"][:400], "\n   type:", r["type"], "\n")

## 5. Keep it running

The service runs while this session is open (Kaggle stops it after at most 12 hours, or soon after you close the tab).
With the ngrok secrets the address stays the same every run, so `backend/.env` never needs to change.